In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [11]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondStructure import FixedRateBondStructure, FixedRateBondStructureFunctionMap
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue, FixedRateBondValueFunctionMap

from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

from RVUtils.Interpolation.GeneralCurveInterpolator import GeneralCurveInterpolator
from RVUtils.ust_viz import plot_usts, plot_usts_comparison
from RVUtils.plt_timeseries import make_secondary_axis_plot

import QuantLib as ql
from BT.misc import ql_cal_date_range

In [12]:
usts_mdp = FixedRateBondsMDP(source="USTS_WEBULL_WSJ_LIVE-RL")
usts_tb = FixedRateBondsTB(usts_mdp)

swaps_mdp = IRSwapsMDP(source="SDR_INTRADAY_RL_USD_SOFR_MTV2_Q12X11")
swaps_tb = IRSwapsTB(swaps_mdp)

In [37]:
start = NY_tz.localize(datetime.datetime(2025, 10, 20, 15, 0))
end = NY_tz.localize(datetime.datetime(2025, 10, 24, 15, 00))
ts_range = ql_cal_date_range(ql.UnitedStates(ql.UnitedStates.GovernmentBond), start=start, end=end, freq="1min") 

usts_df = usts_tb.get_timeseries(
    start=None, end=None,
    timestamps=ts_range,
    queries=[
        FixedRateBondQuery(cusip="Ox430/CT30"),
        # FixedRateBondQuery(cusip="CT10"),
    ])

usts_df

# swaps_df = swaps_tb.get_timeseries(
#     start=None,
#     end=None,
#     timestamps=ts_range,
#     queries=[
#         IRSwapQuery(curve="USD-SOFR-1D", tenor="CT5"),
#         IRSwapQuery(curve="USD-SOFR-1D", tenor="CT10"),
#     ],
#     n_jobs=15,
# )

PRICING FIXED-RATE BONDS.: 100%|██████████| 1920/1920 [00:16<00:00, 115.98it/s]


,Ox430/CT30 CURVE YTM
Date,
2025-10-20 15:00:00-04:00,-1.1
2025-10-21 07:00:00-04:00,-1.1
2025-10-21 07:01:00-04:00,-1.1
2025-10-21 07:02:00-04:00,-1.1
2025-10-21 07:03:00-04:00,-1.1
...,...
2025-10-24 14:56:00-04:00,-1.1
2025-10-24 14:57:00-04:00,-1.1
2025-10-24 14:58:00-04:00,-1.1


In [5]:
# df = swaps_df.join(usts_df)
# df["5Y SPREADS"] = (df["USD-SOFR-1D CT5 OUTRIGHT RATE"] - df["CT5 OUTRIGHT YTM"]) * 100
# df["10Y SPREADS"] = (df["USD-SOFR-1D CT10 OUTRIGHT RATE"] - df["CT10 OUTRIGHT YTM"]) * 100
# df

In [29]:
# plot, fig, ax, ax2, legend = make_secondary_axis_plot(ylabel_left="RATE", ylabel_right="RATE", title=None, engine="plotly")
# plot(
#     df["o10/CT10 CURVE YTM"],
#     which="left",
#     indicators=[
        # {"kind": "last", "show_date": True, "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},
        # {"kind": "sma", "window": 20, "style": {"linestyle": "--", "color": "red"}},
        # {"kind": "simple_avg", "label": "Long-run mean", "style": {"linestyle": "--", "color": "red"}},
        # {"kind": "cum_change", "from_at": "2025-09-24", "label": "Chg from Logan TCGR Speech"},
        # {"kind": "hurst", "hide": True},
        # {"kind": "vol", "returns": "normal", "window": 20, "hide": True},
        # {"kind": "half_life", "method": "ou", "demean": True, "style": {"linestyle": ":"}, "hide": True},
    # ],
    # ou={"enable": True, "steps": 90, "add_metrics_to_legend": True}
# )
# legend(show_date=True)
# plt.show()

plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(usts_df["Ox430/CT30 CURVE YTM"], which="left")
# plot(df["USD-SOFR-1D CT10 OUTRIGHT RATE"], which="left")
# plot(df["USD-SOFR-1D CT5/CT30 CURVE RATE"], which="left")
# plot(df["CT10 OUTRIGHT YTM"], which="right")
legend(valfmt="{:.3f}", show_date=True)

In [27]:
from MDP.FixedRateBonds.WEBULL.WebullFintechFetcher import WebullFintechFetcher

start = CHI_tz.localize(datetime.datetime(2025, 10, 23, 7, 0))
end = CHI_tz.localize(datetime.datetime(2025, 10, 24, 16, 00))

WebullFintechFetcher().intraday_by_tickers(tickers=["TY"], start=start, end=end)

FETCHING FUTURES INTRADAY...: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]


,TY
2025-10-23 07:00:00-05:00,113.500000
2025-10-23 07:01:00-05:00,113.484375
2025-10-23 07:02:00-05:00,113.500000
2025-10-23 07:03:00-05:00,113.500000
2025-10-23 07:04:00-05:00,113.515625
...,...
2025-10-24 15:56:00-05:00,113.390625
2025-10-24 15:57:00-05:00,113.390625
2025-10-24 15:58:00-05:00,113.390625
2025-10-24 15:59:00-05:00,113.390625
